# TrainLM public TPU validation

This notebook validates the **public** TrainLM workflow on a fresh, single-VM TPU runtime. It deliberately does not launch workers, configure ranks, import private coordinator types, or parse stage logs. `TrainLMTrainer` owns those details.

The sequence is: install → configure immutable inputs → validate packed data → allocation-free dry run → TPU smoke → scheduled evaluation/checkpoint run → exact resume → evidence review.

> Run this only in the owner’s Kaggle or Cloud TPU environment. The two execution cells are opt-in because they allocate TPU time.


## 1. Install from the checked-out revision

Select a TPU runtime, open a fresh kernel, check out the branch/commit you intend to validate, and run this cell. Restart the notebook kernel after installation before continuing.


In [ ]:
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt


## 2. Public-run parameters

Set a lowercase 40-character Hugging Face commit SHA—never a mutable branch or tag. The manifest directories must already contain TrainLM packed-bin manifests and their referenced payloads. Keep the execution gate false until the dry run succeeds.


In [ ]:
import json
import os
import re
from pathlib import Path

MODEL_ID = os.environ.get("TRAINLM_MODEL_ID", "microsoft/Phi-3.5-mini-instruct")
MODEL_REVISION = os.environ.get("TRAINLM_MODEL_REVISION", "")
TRAIN_MANIFEST_DIR = Path(os.environ.get("TRAINLM_TRAIN_MANIFEST_DIR", "/kaggle/input/trainlm-packed/train"))
EVAL_MANIFEST_DIR = Path(os.environ.get("TRAINLM_EVAL_MANIFEST_DIR", "/kaggle/input/trainlm-packed/eval"))
OUTPUT_ROOT = Path(os.environ.get("TRAINLM_OUTPUT_ROOT", "/kaggle/working/trainlm-validation"))
SEQUENCE_LENGTH = int(os.environ.get("TRAINLM_SEQUENCE_LENGTH", "2048"))
RUN_TPU = False if "TRAINLM_RUN_TPU" not in os.environ else os.environ["TRAINLM_RUN_TPU"] == "1"

assert re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION), "Set an immutable 40-character MODEL_REVISION."
assert TRAIN_MANIFEST_DIR.is_dir(), TRAIN_MANIFEST_DIR
assert EVAL_MANIFEST_DIR.is_dir(), EVAL_MANIFEST_DIR
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({"model": MODEL_ID, "revision": MODEL_REVISION, "output": str(OUTPUT_ROOT)})


## 3. Validate both packed datasets on the host

Construction eagerly validates manifests, payload sizes, hashes, token bounds, unique shard IDs, and fixed sequence geometry before any TPU worker starts.


In [ ]:
from trainlm import PackedBinDataset, TrainLMTrainer, TrainLMTrainingArguments

train_dataset = PackedBinDataset.from_directory(
    TRAIN_MANIFEST_DIR,
    sequence_length=SEQUENCE_LENGTH,
    split="train",
    seed=42,
)
eval_dataset = PackedBinDataset.from_directory(
    EVAL_MANIFEST_DIR,
    sequence_length=SEQUENCE_LENGTH,
    split="validation",
)
assert len(train_dataset) > 0 and len(eval_dataset) > 0
print({"train_examples": len(train_dataset), "eval_examples": len(eval_dataset)})


## 4. Build one public trainer and inspect the allocation-free plan

This verifies the same facade used for execution. On TPU the detailed model inspection happens inside workers, so the host report should honestly retain that limitation.


In [ ]:
def make_trainer(output_dir, *, max_steps, save_steps=None, eval_steps=None):
    args = TrainLMTrainingArguments(
        output_dir=output_dir,
        max_steps=max_steps,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        sequence_length=SEQUENCE_LENGTH,
        gradient_accumulation_steps=1,
        learning_rate=3e-4,
        bf16=True,
        accelerator="tpu",
        logging_steps=1,
        save_steps=save_steps,
        eval_steps=eval_steps,
        seed=42,
    )
    return TrainLMTrainer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset if eval_steps is not None else None,
    )

trainer = make_trainer(OUTPUT_ROOT / "smoke", max_steps=2)
plan = trainer.explain(format="dict")
print(json.dumps(plan, indent=2, sort_keys=True))
assert plan["backend"] == "xla"
assert plan["selected_path"] == "tpu_coordinator"


## 5. Exercise the installed CLI dry-run

The CLI delegates to the same facade. This validates configuration and datasets but does not allocate a TPU or validate checkpoint contents.


In [ ]:
import yaml

CONFIG_PATH = OUTPUT_ROOT / "validation.yaml"
CONFIG_PATH.write_text(yaml.safe_dump({
    "api_version": "1",
    "model": {
        "provider": "huggingface",
        "initialization": "pretrained",
        "name_or_path": MODEL_ID,
        "revision": MODEL_REVISION,
    },
    "training_args": {
        "output_dir": str(OUTPUT_ROOT / "cli-smoke"),
        "max_steps": 2,
        "sequence_length": SEQUENCE_LENGTH,
        "per_device_train_batch_size": 1,
        "bf16": True,
        "accelerator": "tpu",
        "logging_steps": 1,
    },
}, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text())


In [ ]:
!python -m trainlm train --config "$CONFIG_PATH" --train-manifest-dir "$TRAIN_MANIFEST_DIR" --dry-run


## 6. Two-update TPU smoke (opt-in)

This is the first hardware allocation. The public result includes normalized trainer state, worker summary, and metrics; notebook code does not inspect private stage logs.


In [ ]:
smoke_result = None
if RUN_TPU:
    trainer = make_trainer(OUTPUT_ROOT / "smoke", max_steps=2)
    smoke_result = trainer.train()
    print(json.dumps(smoke_result, indent=2, sort_keys=True, default=str))
    assert smoke_result["trainer_state"]["step"] == 2
    assert smoke_result["trainer_state"]["phase"] == "finalized"
else:
    print("Skipped. Set RUN_TPU = True after dry-run review.")


## 7. Scheduled evaluation and checkpoint lifecycle (opt-in)

Six updates exercise structured callback metrics, evaluation at updates 2/4/6, and committed rank-local checkpoints at updates 2/4/6. A committed checkpoint contains `manifest.json` plus one shard per active rank.


In [ ]:
lifecycle_result = None
LIFECYCLE_DIR = OUTPUT_ROOT / "lifecycle"
if RUN_TPU:
    lifecycle_trainer = make_trainer(
        LIFECYCLE_DIR,
        max_steps=6,
        save_steps=2,
        eval_steps=2,
    )
    lifecycle_result = lifecycle_trainer.train()
    print(json.dumps(lifecycle_result, indent=2, sort_keys=True, default=str))
    checkpoint = LIFECYCLE_DIR / "checkpoint-4"
    assert (checkpoint / "manifest.json").is_file(), checkpoint
    assert lifecycle_result["trainer_state"]["step"] == 6
else:
    print("Skipped. This phase runs only after the smoke passes.")


## 8. Exact resume from a committed checkpoint (opt-in)

Resume uses the public `resume_from_checkpoint` argument. It restores rank-local model, optimizer, scheduler, runtime, trainer, RNG, mesh, and packed-data progress, then advances from checkpoint 4 to update 6 in a separate output directory.


In [ ]:
resume_result = None
if RUN_TPU:
    checkpoint = LIFECYCLE_DIR / "checkpoint-4"
    resume_trainer = make_trainer(
        OUTPUT_ROOT / "resume-from-4",
        max_steps=6,
        save_steps=2,
        eval_steps=2,
    )
    resume_result = resume_trainer.train(resume_from_checkpoint=checkpoint)
    print(json.dumps(resume_result, indent=2, sort_keys=True, default=str))
    assert resume_result["trainer_state"]["step"] == 6
    assert str(checkpoint) == resume_result["worker_summary"]["resumed_from_checkpoint"]
else:
    print("Skipped. Resume requires the committed checkpoint from phase 7.")


## 9. Review public performance and correctness evidence

This cell checks only fields returned by the public facade. A successful smoke is not performance certification: `performance_certified` remains false until the repository’s parity/certification gates consume matched target-hardware evidence.


In [ ]:
if RUN_TPU:
    summary = lifecycle_result["worker_summary"]
    evidence = {
        "world_size": summary["world_size"],
        "steps": summary["steps"],
        "global_supervised_tokens": summary["global_supervised_tokens"],
        "steady_global_supervised_tokens_per_second": summary["steady_global_supervised_tokens_per_second"],
        "performance_certified": summary["performance_certified"],
        "committed_checkpoints": summary["committed_checkpoints"],
    }
    print(json.dumps(evidence, indent=2, sort_keys=True))
    assert evidence["world_size"] == summary["expected_world_size"]
    assert evidence["performance_certified"] is False
else:
    print("No TPU evidence yet.")


## Target-hardware acceptance checklist

Record the checked-out commit and attach the public result artifacts before reporting a pass.

- [ ] Host dataset validation and CLI dry-run succeed.
- [ ] The two-update smoke finalizes at the expected world size with finite loss.
- [ ] Scheduled evaluation metrics appear at updates 2, 4, and 6.
- [ ] Checkpoints 2, 4, and 6 are committed with every rank shard.
- [ ] Resume from checkpoint 4 reaches update 6 without topology or data-position errors.
- [ ] Global supervised-token throughput is present and warm-up semantics are recorded.
- [ ] `performance_certified` remains false until matched parity, stability, numerical-alignment, export, and scaling evidence passes the release gates.

Current boundary: public TPU `save_model()`/`save_state()` remain intentionally unavailable; canonical HF export is worker-owned and release certification is a separate measured workflow.
